<a href="https://colab.research.google.com/github/Tamur-Naseem/FLY-RANK/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tamur-Naseem/FLY-RANK/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window
Unit of Analysis: One row represents a single pseudonymized content item (page) observed on one specific day (content_hash_id + report_date).
Time Window: The feature engineering and training window is restricted to March 2026 (month=2026-03). This mid-panel month is chosen to ensure a clean separation from the final testing month (June 2026), strictly preventing temporal leakage.

*One row = one what, over which dates? State it, then verify it below.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*
Features: impressions_90d, clicks_90d, avg_position, content_age_days. (All are strictly measurable at the decision moment).

Label (Proxy): is_opportunity. A binary flag (1/0) defined as having established traffic (impressions_90d > 500) but showing a trend_direction of 'down'.

Context: content_hash_id (page identifier) and report_date (temporal anchor).

Excluded: trend_direction (from features), raw URL, and raw page text. trend_direction is excluded from the feature set because it is mathematically derived to create the label; including it would result in terminal data leakage where the model "cheats" by seeing the future outcome.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*
The following code verifies the data contract by confirming the daily grain, checking the date bounds for the mid-panel month, and validating data availability.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import duckdb
import os
from google.colab import userdata # <--- THIS IMPORT IS REQUIRED
from huggingface_hub import hf_hub_download

# 1. Download the data slice locally using the huggingface_hub library
file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=userdata.get('HF_TOKEN')
)

# 2. Use duckdb to query the local file
con = duckdb.connect()

# Fact 1: Grain verification
print("Fact 1: Grain (Rows vs Unique Pages)")
display(con.execute(f"SELECT COUNT(*) as total_rows, COUNT(DISTINCT content_hash_id) as unique_pages FROM '{file_path}'").df())

# Fact 2: Date window verification
print("\nFact 2: Date Span")
display(con.execute(f"SELECT MIN(report_date) as start_date, MAX(report_date) as end_date FROM '{file_path}'").df())

# Fact 3: Availability check
print("\nFact 3: Data Availability (IS TRUE)")
display(con.execute(f"SELECT COUNT(*) as valid_rows FROM '{file_path}' WHERE ga4_data_available IS TRUE").df())


fact_content_daily_performance/month=202(…):   0%|          | 0.00/124M [00:00<?, ?B/s]

Fact 1: Grain (Rows vs Unique Pages)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_pages
0,9841378,331437



Fact 2: Date Span


,start_date,end_date
0,2026-03-01,2026-03-31



Fact 3: Data Availability (IS TRUE)


,valid_rows
0,413966


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*
Data Limits: This slice cannot distinguish between true content decay and seasonal cycles. Because our proxy relies on a 90-day window, a traffic drop in March could be a normal seasonal pattern for specific categories (e.g., winter content) rather than a signal for a refresh. Furthermore, historical data points may have ga4_data_available as FALSE for older records, meaning some engagement features will be systematically null in this mid-panel slice.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.